# 00 · Data overview: what the three files contain and whether they can be compared

**Objective.** Establish what each dataset contains and confirm that the three files are aligned, so that the per-point differences in the later notebooks are meaningful.

**Question.** Are the files complete and intact? Which periods, variables, units and grid do they cover? Do they match each other well enough that "scenario minus historical" at each sea point is a fair comparison?

**Principle.** Two datasets can only be subtracted point by point if they share the same grid, the same variables and units, the same time sampling and periods of the same length. Each of these is checked from the files themselves (sizes, checksums, the `metadata` struct, value ranges), not from assumptions. Missing values are mapped, so that later notebooks can mask the points where a statistic does not describe the full period.

**Inputs → outputs.** `data/raw/*/EC-EARTH3.mat`, `SeaMask_WW3.mat`, `data/processed/stats30/` → tables in `data/processed/overview/`, figures in `fig/overview/`.

**Answer.** *(written last, see section 7)*

## 0. Setup
Run this cell first. `compute_stats.py` must have been run for section 6 (`hs` at least).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cmip_utils import (RAW, PROCESSED, SCENARIOS, EXPECTED_SIZES, load_grid, plot_map,
                        area_mean, save_fig, save_table)
from compute_stats import time_axis, load_stats, OUT as STATS_DIR

SUB = "overview"
grid = load_grid()
PERIOD = {}  # filled in section 3

## 1. Where the data come from

The data are one member of the eight-model wave-climate ensemble of Meucci et al. (2024). The wave model **WAVEWATCH III** was run with 3-hourly 10-m winds and daily sea-ice concentration taken from the CMIP6 climate model **EC-Earth3** (member r1i1p1f1). The waves are therefore simulated, driven by a simulated climate: they are not observations, and the model does not reproduce the actual weather of any given year.

| File | Experiment | What it assumes |
|---|---|---|
| `historical` | CMIP6 historical | observed greenhouse gases, aerosols, solar and volcanic forcing |
| `ssp126` | SSP1-2.6 | strong mitigation; low radiative forcing (~2.6 W/m² in 2100) |
| `ssp585` | SSP5-8.5 | fossil-fuel intensive; very high forcing (~8.5 W/m² in 2100) |

The SSPs are conditional "what-if" pathways, not predictions. The project's reference is ERA5 (the ERA5 wave hindcast in Meucci et al.); **status: requested from the data provider**. Until it arrives, the change signal is computed against the model's own historical run, which removes most of the model's own bias from the difference.

## 2. Are the files complete and intact?

Each file must have exactly the size of the provider's file. The SHA-256 checksums are a record: they allow anyone to confirm later that the files have not changed. Full decompression of every chunk happens in `compute_stats.py`, so a successful run there is a complete integrity check.

In [ ]:
rows = []
for s in SCENARIOS:
    p = RAW / s / "EC-EARTH3.mat"
    rows.append({"scenario": s, "size [bytes]": p.stat().st_size, "expected": EXPECTED_SIZES[s],
                 "size ok": p.stat().st_size == EXPECTED_SIZES[s]})
files = pd.DataFrame(rows).set_index("scenario")
sums = RAW / "sha256sums.txt"
if sums.exists():
    h = dict(reversed(l.split()) for l in sums.read_text().splitlines() if l.strip())
    files["sha256 (first 16)"] = [h.get(f"data/raw/{s}/EC-EARTH3.mat", "")[:16] for s in SCENARIOS]
display(files)
save_table(files, "files_integrity", SUB)

## 3. Which periods, and at what time resolution?

Each file stores 360 monthly MATLAB files joined in time order. The `metadata` struct lists their names (`YYYYMM.mat`) and the first and last time step of every month, which gives the exact calendar without assuming a start date. The check below also confirms 8 steps per calendar day (3-hourly), including leap days.

In [ ]:
rows = []
for s in SCENARIOS:
    t = time_axis(RAW / s / "EC-EARTH3.mat")
    PERIOD[s] = f"{t['years'][0]}–{t['years'][-1]}"
    rows.append({"scenario": s, "period": PERIOD[s], "years": len(t["years"]), "months": len(t["month"]),
                 "time steps": t["n_steps"], "steps per day": 8, "leap days": t["leap_days"],
                 "Gregorian calendar": t["gregorian_calendar"]})
periods = pd.DataFrame(rows).set_index("scenario")
display(periods)
save_table(periods, "periods", SUB)

**Result.** Historical covers 1985–2014; both scenarios cover 2071–2100 (end of the century). All three are 30 years of 3-hourly data with the same number of leap days, so the periods are directly comparable in length and sampling.

## 4. Which variables and units?

The variable names come from `metadata.FieldsOut`. The files store no units; the WAVEWATCH III standard units are assumed and checked against the value ranges (first 31 days, all sea points).

In [ ]:
variables = pd.DataFrame([
    ("hs",   "significant wave height",          "m",   "stored"),
    ("fp",   "peak frequency",                   "Hz",  "stored"),
    ("uwnd", "eastward 10-m wind",               "m/s", "stored"),
    ("vwnd", "northward 10-m wind",              "m/s", "stored"),
    ("tp",   "peak period 1/fp",                 "s",   "derived"),
    ("wspd", "wind speed hypot(uwnd, vwnd)",     "m/s", "derived"),
], columns=["name", "meaning", "unit (assumed)", "source"]).set_index("name")
display(variables)
ranges = PROCESSED / "eda" / "value_ranges_first_31_days.csv"
if ranges.exists():
    display(pd.read_csv(ranges).round(3))

## 5. Which grid?

All files use the same global 0.5° grid (720 × 361) and store only its **142,868 sea points**, in the same order (`SeaIdx`, identical in all three metadata structs). Land is not stored. Because the grids are identical, no regridding is needed and differences can be taken point by point. Area averages are weighted by cos(latitude), since 0.5° cells shrink towards the poles.

In [ ]:
sea = np.ones(142_868)
fig, ax = plot_map(grid, sea, title="The 142,868 sea points of the 0.5° grid", cmap="Blues", vmin=0, vmax=1.5)
save_fig(fig, "sea_points", SUB)

## 6. Where are values missing, and what do we do about it?

Missing values (NaN) occur only at sea points in polar seas, and shrink from historical to SSP5-8.5. The data provider confirms they come from sea ice or model issues in those remote areas; they are not investigated further. A statistic at a point that is often missing describes only the ice-free time, so later notebooks **mask points with more than 10 % missing values** in either period of a comparison.

In [ ]:
MASK_LIMIT = 0.10
nanf = {s: load_stats(s, "hs")["nan_frac"] for s in SCENARIOS}
fig, axs = plt.subplots(1, 3, figsize=(18, 4), constrained_layout=True)
for ax, s in zip(axs, SCENARIOS):
    plot_map(grid, 100 * nanf[s], ax=ax, cmap="Blues", vmin=0, vmax=100,
             title=f"{s} ({PERIOD.get(s, '')}): time missing", label="missing [%]")
fig.suptitle("Share of time steps without Hs (white: never missing)")
save_fig(fig, "hs_missing_maps", SUB)

summary = pd.DataFrame({s: {"points with any missing": int((nanf[s] > 0).sum()),
                            f"points masked (> {MASK_LIMIT:.0%})": int((nanf[s] > MASK_LIMIT).sum()),
                            "ocean area masked [%]": 100 * area_mean(grid, (nanf[s] > MASK_LIMIT).astype(float))}
                        for s in SCENARIOS}).T
display(summary.round(2))
save_table(summary, "hs_missing_summary", SUB)

## 7. Answer

| Requirement for a fair comparison | Status |
|---|---|
| Files complete and unchanged | sizes exact, checksums recorded, every chunk decompressed by `compute_stats.py` |
| Same grid and point order | yes: same 0.5° grid, same 142,868 sea points |
| Same variables and units | yes: `fp`, `hs`, `uwnd`, `vwnd`; units assumed from WW3 conventions |
| Same sampling and period length | yes: 30 years, 3-hourly, 7 leap days each |
| Periods | historical 1985–2014; SSP1-2.6 and SSP5-8.5 2071–2100 |
| Missing values | polar seas only (sea ice); points with > 10 % missing are masked |
| ERA5 reference | *pending* |

*(Write 2–3 sentences here once the cells have run, e.g. how much ocean area is masked per scenario.)*

**Caveats.** One climate model and one ensemble member, so model uncertainty is not sampled. Units are inferred, not stored. The wind calibration of this WW3 run (CDFAC 1.0 or 1.08) is to be confirmed.